# Fine-tuning de Qwen3.5-4B para Minimarket

Aplicación para construir un agente de atención por WhatsApp que debe conocer el catálogo real de
250 productos, el horario, los medios de pago y la política de devoluciones del dossier.

Evaluamos dos formas de aplicar LoRA — solo en las capas de atención, o en atención + feed-forward — para entender cuál generaliza mejor sin memorizar de más. Los demás hiperparámetros de entrenamiento se mantienen fijos.


In [ ]:
# Unsloth trae ya fijadas las versiones de transformers/trl/peft compatibles
# con Qwen3.5, por eso no se instalan por separado.
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo


In [ ]:
import gc
import json
import os
import random

import torch
from unsloth import is_bfloat16_supported

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM total (GB):", round(props.total_memory / 1e9, 2))
    print("Esta GPU soporta bf16:", is_bfloat16_supported())
else:
    raise RuntimeError("No hay GPU activa. En Colab: Entorno de ejecución > Cambiar tipo de entorno > GPU")


## Configuración

Dos grupos de valores: las rutas y el modelo (fijos), y los hiperparámetros de
entrenamiento — `LORA_DROPOUT`, `LEARNING_RATE` y el resto (epochs, batch, `r`,
warmup, weight decay, `lora_alpha`) — que se mantienen **fijos en todo el notebook**,
agrupados en `CONFIG_BASE` más adelante. Lo único que se compara entre las dos
configuraciones de este notebook es dónde se inserta LoRA (`target_modules`).


In [ ]:
# Si tu notebook corre desde la carpeta notebook/ de tu repositorio (clonado o
# montado desde Drive), estas rutas relativas apuntan directo a data/.
# Si tus archivos tienen otro nombre, ajústalos aquí.
RUTA_DATASET = "/content/sample_data/data/dataset_minimarket.jsonl"
RUTA_DOSSIER = "/content/sample_data/data/dossier_minimarket.md"

MODELO_INSTRUCT = "unsloth/Qwen3.5-4B"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # QLoRA: el modelo base queda congelado y cuantizado a 4 bits;
                      # los adaptadores LoRA se entrenan aparte, siempre en 16 bits

# Hiper-parametros fijos a lo largo del fine -tuning

LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4

SEMILLA = 3407
random.seed(SEMILLA)

# Con esto en True se reduce el dataset de entrenamiento a un puñado de ejemplos,
# útil para revisar que todo el pipeline corre de punta a punta antes de lanzar
# la ronda completa de experimentos (que toma bastante más tiempo).
MODO_RAPIDO = False


## 1. Datos: carga y separación en train / validation / test



In [ ]:
from datasets import load_dataset
from collections import Counter

dataset_crudo = load_dataset("json", data_files=RUTA_DATASET, split="train")

print("Conversaciones totales:", len(dataset_crudo))
print("Por categoría:")
for categoria, cantidad in Counter(dataset_crudo["categoria"]).most_common():
    print(f"  {categoria}: {cantidad}")


In [ ]:
# Separación 80% train / 10% validation / 10% test, estratificada informalmente
# por categoría (barajamos con semilla fija para que sea reproducible).

split_1 = dataset_crudo.train_test_split(test_size=0.20, seed=SEMILLA)

dataset_train = split_1["train"]

split_2 = split_1["test"].train_test_split(test_size=0.50, seed=SEMILLA)
dataset_val = split_2["train"]
dataset_test = split_2["test"]

print("Train:", len(dataset_train))
print("Validation:", len(dataset_val))
print("Test:", len(dataset_test))

if MODO_RAPIDO:
    dataset_train = dataset_train.select(range(min(40, len(dataset_train))))
    dataset_val = dataset_val.select(range(min(15, len(dataset_val))))
    print("MODO_RAPIDO activo, usando solo:", len(dataset_train), "train /", len(dataset_val), "val")


In [ ]:
# El dossier no se usa para entrenar, se usa como referencia para las preguntas
# de control más abajo (horario, productos, política de devoluciones).

with open(RUTA_DOSSIER, encoding="utf-8") as f:
    dossier = f.read()

print("Dossier cargado,", len(dossier.splitlines()), "líneas")


## 2. Cargar el modelo con QLoRA

`load_in_4bit=True` los pesos del modelo base quedan cuantizados en 4 bits, congelados: no se
tocan durante el entrenamiento. Sobre eso se van a insertar los adaptadores LoRA.


In [ ]:
from unsloth import FastLanguageModel

modelo_base, tokenizer_base = FastLanguageModel.from_pretrained(
    model_name=MODELO_INSTRUCT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=False,
)
FastLanguageModel.for_inference(modelo_base)

print("Modelo cargado:", round(modelo_base.num_parameters() / 1e9, 2), "B parámetros")


In [ ]:
from contextlib import nullcontext


def plantilla_chat(tokenizer, mensajes, add_generation_prompt=True):
    return tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=add_generation_prompt,
        enable_thinking=False,
    )


def responder(modelo, tokenizer, pregunta, con_lora=True, max_new_tokens=300):
    """Genera una respuesta del agente. con_lora=False apaga los adaptadores con
    disable_adapter() de PEFT, sin recargar el modelo, para comparar sobre exactamente
    el mismo modelo base.

    Los parámetros de generación (temperature=0.2, repetition_penalty=1.2) son bajos
    a propósito: para un agente de atención al cliente interesa que sea consistente y
    poco creativo con los datos, no que varíe la respuesta cada vez que le preguntas lo mismo.
    """
    prompt = plantilla_chat(tokenizer, [{"role": "user", "content": pregunta}])
    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")

    if con_lora or not hasattr(modelo, "disable_adapter"):
        contexto = nullcontext()
    else:
        contexto = modelo.disable_adapter()

    with contexto:
        salida = modelo.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            repetition_penalty=1.2,
            top_p=0.9,
            use_cache=True,
        )
    return tokenizer.decode(salida[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


## 3. Preguntas de control

Dos grupos. El primero son preguntas normales de negocio, con respuesta correcta verificable
en el dossier. El segundo grupo son preguntas para detectar dos problemas distintos:

- **¿La tienda abre el 25 de diciembre?** el dossier no menciona el 25 de diciembre en
  específico, pero sí tiene una regla general para "domingos y feriados". Para responder
  bien, el modelo tiene que darse cuenta de que el 25 de diciembre es un feriado y aplicar
  esa regla.
- **¿Dónde quedan sus otras sedes?** el dossier describe una sola tienda. La respuesta
  correcta es decir que no hay otras sedes. Si el modelo con un tono seguro se inventa una dirección de una sucursal que no existe, es una
  señal de que el fine-tuning le enseñó a sonar seguro incluso inventando.
- **¿Qués es Teoría de Juegos y Organización Industrial, en dos líneas?** esta es la prueba de olvido catastrófico en
  el sentido estricto: una pregunta sin ninguna relación con el minimarket. Si el modelo
  fine-tuned responde peor que el modelo sin LoRA, es señal de que el entrenamiento afectó
  capacidades generales que no tenían nada que ver con el dominio.


In [ ]:
PREGUNTA_QUEJA_INCOMPLETO = (
    "Hola, mi pedido llegó incompleto, pedí Inca Kola 500ml y Yogurt Gloria Fresa 130g "
    "y no me llegó ninguno de los dos, qué pasó?"
)
PREGUNTA_HORARIO = "Buenas, cuál es su horario de atención?"
PREGUNTA_CATALOGO = "Qué tipo de productos venden ahí?"

# fuera de lo que el modelo vio en el dataset de entrenamiento
PREGUNTA_FERIADO = "Van a abrir el 25 de diciembre?"
PREGUNTA_OTRAS_SEDES = "Dónde quedan sus otras sedes o locales?"
PREGUNTA_CONOCIMIENTO_GENERAL = "Explícame en dos líneas qué Tería de Juegos y Organziación Industrial"

PREGUNTAS_NEGOCIO = [PREGUNTA_QUEJA_INCOMPLETO, PREGUNTA_HORARIO, PREGUNTA_CATALOGO]
PREGUNTAS_CONTROL = [PREGUNTA_FERIADO, PREGUNTA_OTRAS_SEDES, PREGUNTA_CONOCIMIENTO_GENERAL]


In [ ]:
# Respuestas del modelo instruct antes de cualquier fine-tuning, para tener con qué comparar
# más adelante. En este punto disable_adapter() ni siquiera existe todavía (no hay LoRA
# aplicado), así que con_lora no cambia nada aquí.
respuestas_base = {}
for pregunta in PREGUNTAS_NEGOCIO + PREGUNTAS_CONTROL:
    respuestas_base[pregunta] = responder(modelo_base, tokenizer_base, pregunta, con_lora=False)
    print("PREGUNTA:", pregunta)
    print("RESPUESTA:", respuestas_base[pregunta])
    print()


## 4. Formatear las conversaciones para entrenamiento

Se aplica la plantilla de chat de Qwen3.5 a cada
conversación, sin generar nada nuevo (`add_generation_prompt=False`), para que el texto
quede tal cual lo va a ver el modelo durante el entrenamiento.


In [ ]:
def formatear_conversaciones(tokenizer):
    def _mapear(examples):
        return {"text": [
            plantilla_chat(tokenizer, conv, add_generation_prompt=False)
            for conv in examples["conversations"]
        ]}
    return _mapear


def preparar_split(dataset_split, tokenizer):
    formateado = dataset_split.map(formatear_conversaciones(tokenizer), batched=True)
    columnas_sobrantes = [c for c in formateado.column_names if c != "text"]
    return formateado.remove_columns(columnas_sobrantes)


dataset_train_texto = preparar_split(dataset_train, tokenizer_base)
dataset_val_texto = preparar_split(dataset_val, tokenizer_base)
dataset_test_texto = preparar_split(dataset_test, tokenizer_base)

print(dataset_train_texto[0]["text"][:500])


## 5. Diseño experimental — ¿dónde aplicar LoRA?

Los hiperparámetros de entrenamiento (epochs, batch, `r`, warmup, weight decay,
`lora_alpha`) se mantienen fijos en `CONFIG_BASE`, con `lora_alpha = r` por
convención. Lo único que se compara en este notebook es dónde se inserta LoRA:
solo en las proyecciones de atención (`q_proj, k_proj, v_proj, o_proj`) contra
también en las capas feed-forward / MLP (`gate_proj, up_proj, down_proj`). Más
capas adaptadas dan más capacidad de ajuste, pero también más parámetros
entrenables y más riesgo de sobreajuste con un dataset de este tamaño.


In [ ]:
ATENCION = ["q_proj", "k_proj", "v_proj", "o_proj"]
ATENCION_Y_MLP = ATENCION + ["gate_proj", "up_proj", "down_proj"]

CONFIG_BASE = {
    "epochs": 3,
    "per_device_batch": 2,
    "grad_accum": 4,
    "r": 16,
    "lora_alpha": 16,
    "warmup_steps": 10,
    "weight_decay": 0.01,
}

if MODO_RAPIDO:
    CONFIG_BASE = {**CONFIG_BASE, "epochs": 1}

print(CONFIG_BASE)


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only


def entrenar_config(config, dataset_train_texto, dataset_val_texto, tag=""):
    """Carga una copia limpia del modelo instruct, le aplica LoRA con la configuración
    dada, entrena, y devuelve el trainer entrenado junto con su historial de loss.
    Recargar el modelo en cada llamada evita que los adaptadores de una corrida anterior
    se mezclen con la siguiente.
    """
    modelo, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODELO_INSTRUCT,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=LOAD_IN_4BIT,
        full_finetuning=False,
    )

    modelo = FastLanguageModel.get_peft_model(
        modelo,
        r=config["r"],
        target_modules=config["target_modules"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEMILLA,
        max_seq_length=MAX_SEQ_LENGTH,
    )

    dtype_pesos = next(modelo.parameters()).dtype
    usar_bf16 = (dtype_pesos == torch.bfloat16) and is_bfloat16_supported()
    usar_fp16 = (dtype_pesos == torch.float16)

    trainer = SFTTrainer(
        model=modelo,
        tokenizer=tokenizer,
        train_dataset=dataset_train_texto,
        eval_dataset=dataset_val_texto,
        args=SFTConfig(
            max_seq_length=MAX_SEQ_LENGTH,
            per_device_train_batch_size=config["per_device_batch"],
            per_device_eval_batch_size=config["per_device_batch"],
            gradient_accumulation_steps=config["grad_accum"],
            warmup_steps=config["warmup_steps"],
            num_train_epochs=config["epochs"],
            learning_rate=LEARNING_RATE,
            logging_steps=1,
            eval_strategy="epoch",
            optim="adamw_8bit",
            weight_decay=config["weight_decay"],
            lr_scheduler_type="linear",
            seed=SEMILLA,
            output_dir=f"outputs_{config['nombre']}{tag}",
            report_to="none",
            bf16=usar_bf16,
            fp16=usar_fp16,
            dataset_num_proc=1,
        ),
    )

    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    resultado = trainer.train()

    historial_train = [h for h in trainer.state.log_history if "loss" in h and "eval_loss" not in h]
    historial_eval = [h for h in trainer.state.log_history if "eval_loss" in h]

    salida = {
        "config": config,
        "modelo": modelo,
        "tokenizer": tokenizer,
        "trainer": trainer,
        "train_loss": [h["loss"] for h in historial_train],
        "train_steps": [h["step"] for h in historial_train],
        "eval_loss": [h["eval_loss"] for h in historial_eval],
        "eval_epoch": [h["epoch"] for h in historial_eval],
        "tiempo_seg": resultado.metrics["train_runtime"],
        "eval_loss_final": historial_eval[-1]["eval_loss"] if historial_eval else None,
    }
    return salida


def liberar(resultado_entrenamiento):
    """Suelta el modelo y el trainer de un resultado para no acumular VRAM entre corridas.
    Se guardan aparte los números que sí interesa conservar antes de llamar a esto.
    """
    del resultado_entrenamiento["modelo"]
    del resultado_entrenamiento["trainer"]
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
config_solo_atencion = {**CONFIG_BASE, "nombre": "solo_atencion", "target_modules": ATENCION}
config_atencion_mlp = {**CONFIG_BASE, "nombre": "atencion_mlp", "target_modules": ATENCION_Y_MLP}

resultados = {}
for config in [config_solo_atencion, config_atencion_mlp]:
    print("Entrenando configuración:", config["nombre"])
    resultado = entrenar_config(config, dataset_train_texto, dataset_val_texto)
    print(f"  tiempo: {resultado['tiempo_seg']:.1f}s, eval_loss final: {resultado['eval_loss_final']:.4f}")
    resultados[config["nombre"]] = {
        k: v for k, v in resultado.items() if k not in ("modelo", "trainer")
    }
    liberar(resultado)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4.5))
for nombre, r in resultados.items():
    plt.plot(r["train_steps"], r["train_loss"], marker="o", markersize=2, label=nombre)
plt.xlabel("Paso de entrenamiento")
plt.ylabel("Loss (train)")
plt.title("Solo atención vs. atención + feed-forward (MLP)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for nombre, r in resultados.items():
    print(nombre, "- eval_loss final:", round(r["eval_loss_final"], 4))

mejor_nombre = min(resultados, key=lambda n: resultados[n]["eval_loss_final"])
mejor_config_final = config_solo_atencion if mejor_nombre == "solo_atencion" else config_atencion_mlp
mejor_config_final = {**mejor_config_final, "nombre": "final"}
print()
print("Mejor alcance de LoRA:", mejor_nombre)


## 6. Entrenamiento final y evaluación contra el test set

Se reentrena una última vez con la configuración ganadora del diseño experimental
anterior. Este modelo es el que se evalúa contra el test set.


In [ ]:
print("Configuración final elegida:")
print(mejor_config_final)

resultado_final = entrenar_config(mejor_config_final, dataset_train_texto, dataset_val_texto, tag="_final")
modelo_final = resultado_final["modelo"]
tokenizer_final = resultado_final["tokenizer"]
trainer_final = resultado_final["trainer"]

print(f"Eval loss final (validation): {resultado_final['eval_loss_final']:.4f}")


In [ ]:
metricas_test = trainer_final.evaluate(eval_dataset=dataset_test_texto)
print("Loss sobre el test set:", metricas_test["eval_loss"])


## 7. Comparación cualitativa: antes y después del fine-tuning

Para las preguntas de negocio se espera que el modelo
fine-tuned cite datos exactos del dossier (el código de producto, el horario, la política).
Para las preguntas de control, lo deseable no es necesariamente una respuesta larga: es que
el modelo no invente con seguridad algo que no está en el dossier.


In [ ]:
FastLanguageModel.for_inference(modelo_final)

print("### Preguntas de negocio ###\n")
for pregunta in PREGUNTAS_NEGOCIO:
    sin_lora = responder(modelo_final, tokenizer_final, pregunta, con_lora=False)
    con_lora = responder(modelo_final, tokenizer_final, pregunta, con_lora=True)
    print("PREGUNTA:", pregunta)
    print("SIN LoRA:", sin_lora)
    print("CON LoRA:", con_lora)
    print()

print("### Preguntas de control (generalización y olvido catastrófico) ###\n")
for pregunta in PREGUNTAS_CONTROL:
    sin_lora = responder(modelo_final, tokenizer_final, pregunta, con_lora=False)
    con_lora = responder(modelo_final, tokenizer_final, pregunta, con_lora=True)
    print("PREGUNTA:", pregunta)
    print("SIN LoRA:", sin_lora)
    print("CON LoRA:", con_lora)
    print()


Al leer las respuestas de arriba, conviene revisar en concreto:

- En la pregunta del feriado, si el modelo fine-tuned aplica la regla de "domingos y
  feriados" al 25 de diciembre, o si solo repite el horario de lunes a sábado porque
  memorizó esa frase literal.
- En la de otras sedes, si el modelo dice honestamente que hay una sola tienda, o si
  inventa una dirección con el mismo tono seguro que aprendió para el resto de respuestas.
- En la pregunta de Teoría de Juego y Organización Industrial, si la respuesta se mantiene razonable o si el modelo insiste en
  llevar todo al tono de agente de minimarket (señal de sobreajuste de tono).


## 8. Guardar el modelo

Se guardan solo los adaptadores LoRA, que es lo que realmente se entrenó: el modelo base se
puede volver a descargar de Hugging Face en cualquier momento, pero el adaptador es el
resultado propio de este notebook.


In [ ]:
RUTA_LORA = "minimarket_lora"

modelo_final.save_pretrained(RUTA_LORA)
tokenizer_final.save_pretrained(RUTA_LORA)

tamano_bytes = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fs in os.walk(RUTA_LORA) for f in fs
)
print("Adaptadores guardados en:", RUTA_LORA)
print("Tamaño total:", round(tamano_bytes / 1e6, 1), "MB")


In [ ]:
# Opcional: modelo completo mergeado (pesos base + LoRA en un solo checkpoint),
# más pesado pero no necesita PEFT para cargarse. Descomentar si lo vas a desplegar así.

# modelo_final.save_pretrained_merged(
#     "minimarket_la_esquina_merged",
#     tokenizer_final,
#     save_method="merged_16bit",
# )
